# Robust Zero-Sum MARL — Colab Training (10 Parallel Envs)

This notebook trains:
1. **Phase 1** — Nominal DDPG policy (`pi_opt`) on Ant-v5
2. **Phase 2** — Adversarial DDPG (`pi_rob` + `pi_adv`) with transformer disturbance detector

Uses 10 parallel environments via Gymnasium `AsyncVectorEnv` for ~10x faster data collection.

> **Runtime**: Go to *Runtime > Change runtime type* and select **GPU** (T4 or better).

## 1. Setup

In [ ]:
!pip install -q "gymnasium[mujoco]" torch matplotlib pandas

In [ ]:
import os

REPO_DIR = "robust-zero-sum-marl"
if not os.path.isdir(REPO_DIR):
    # Replace with your repo URL
    !git clone https://github.com/alibaniasad1999/robust-zero-sum-marl.git

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
import torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 2. Phase 1 — Train Nominal Policy (pi_opt)

In [ ]:
# ── Training configuration ──────────────────────────────────────
ENV_ID = "Ant-v5"           # MuJoCo environment
NUM_ENVS = 10               # parallel environments
EPOCHS = 100                # total epochs
STEPS_PER_EPOCH = 4000      # transitions per epoch
BATCH_SIZE = 1024           # SGD mini-batch size
START_STEPS = 10000         # random exploration steps
UPDATE_AFTER = 10000        # start SGD after this many transitions
SEED = 0
DEVICE = "auto"             # "auto" picks cuda if available
LOG_DIR = "logs/nominal"

In [ ]:
import gymnasium as gym
from src.agents.ddpg import DDPGAgent

env_fn = lambda: gym.make(ENV_ID)

agent = DDPGAgent(
    env_fn,
    seed=SEED,
    epochs=EPOCHS,
    steps_per_epoch=STEPS_PER_EPOCH,
    batch_size=BATCH_SIZE,
    start_steps=START_STEPS,
    update_after=UPDATE_AFTER,
    num_envs=NUM_ENVS,
    device=DEVICE,
    log_dir=LOG_DIR,
)

agent.train()
agent.save()
print("Phase 1 complete.")

## 3. Phase 2 — Train Adversarial Policy (pi_rob + pi_adv + transformer)

In [ ]:
# ── Adversarial training configuration ───────────────────────────
ADV_EPOCHS = 100
DISTURBANCE_RATIO = 0.1     # adversary action scale vs. protagonist
DISTURBANCE_PROB = 0.5      # probability of disturbance per episode
PI_OPT_PATH = "logs/nominal/checkpoints"  # from Phase 1
ADV_LOG_DIR = "logs/adversarial"

In [ ]:
from src.agents.ddpg import AdversarialDDPGAgent

adv_agent = AdversarialDDPGAgent(
    env_fn,
    seed=SEED,
    epochs=ADV_EPOCHS,
    steps_per_epoch=STEPS_PER_EPOCH,
    batch_size=BATCH_SIZE,
    start_steps=START_STEPS,
    update_after=UPDATE_AFTER,
    num_envs=NUM_ENVS,
    device=DEVICE,
    log_dir=ADV_LOG_DIR,
    disturbance_ratio=DISTURBANCE_RATIO,
    disturbance_probability=DISTURBANCE_PROB,
    use_transformer=True,
    pi_opt_path=PI_OPT_PATH,
)

adv_agent.train()
adv_agent.save()
print("Phase 2 complete.")

## 4. Plot Training Curves

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (label, log_dir) in zip(axes, [
    ("Nominal (pi_opt)", LOG_DIR),
    ("Adversarial (pi_rob)", ADV_LOG_DIR),
]):
    csv_path = f"{log_dir}/progress.csv"
    try:
        df = pd.read_csv(csv_path)
        col = [c for c in df.columns if "return" in c.lower() or "reward" in c.lower()]
        if col:
            ax.plot(df[col[0]], linewidth=0.8)
            ax.set_ylabel(col[0])
        else:
            ax.plot(df.iloc[:, 1], linewidth=0.8)
            ax.set_ylabel(df.columns[1])
        ax.set_xlabel("Epoch")
        ax.set_title(label)
        ax.grid(True, alpha=0.3)
    except FileNotFoundError:
        ax.set_title(f"{label} (no data yet)")

plt.tight_layout()
plt.savefig("training_curves.png", dpi=150)
plt.show()

## 5. Download Checkpoints

In [ ]:
import shutil

# Bundle all checkpoints into a zip
shutil.make_archive("rzsm_checkpoints", "zip", ".", "logs")
print("Created rzsm_checkpoints.zip")

# Colab: trigger download
try:
    from google.colab import files
    files.download("rzsm_checkpoints.zip")
except ImportError:
    print("Not running in Colab — find rzsm_checkpoints.zip in the working directory.")